In [2]:
import os
import pandas as pd
import json

In [3]:
# Define the path to the experiments directory
experiments_dir = "experiments"

# List to store the results for each sub-directory
results = []

# Ensure the experiments directory exists before running
if os.path.exists(experiments_dir):
    # Iterate through each item in the experiments directory
    for entry in os.listdir(experiments_dir):
        sub_dir = os.path.join(experiments_dir, entry)
        
        # Check if the current item is actually a directory
        if os.path.isdir(sub_dir):
            metrics_file = os.path.join(sub_dir, "training_metrics.csv")
            hp_file = os.path.join(sub_dir, "hyperparameters.json")
            
            # Check if the metrics CSV exists in this sub-directory
            if os.path.exists(metrics_file):
                try:
                    df = pd.read_csv(metrics_file)
                    
                    # Verify that the required columns exist
                    required_cols = ['frame_f1', 'mean_iou', 'segment_f1']
                    if all(col in df.columns for col in required_cols):
                        
                        # Determine the best epoch by summing the 3 metrics (Change this math if you want different weights)
                        df['combined_score'] = df['frame_f1'] + df['mean_iou'] + df['segment_f1']
                        
                        # Find the index of the row with the highest combined score
                        best_idx = df['combined_score'].idxmax()
                        best_row = df.iloc[best_idx]
                        
                        # Get the epoch number (Assumes an 'epoch' column exists, otherwise uses row index + 1)
                        best_epoch = int(best_row['epoch']) if 'epoch' in df.columns else best_idx + 1
                        
                        frame_f1 = best_row['frame_f1']
                        mean_iou = best_row['mean_iou']
                        segment_f1 = best_row['segment_f1']
                        
                        # Look for the description in hyperparameters.json
                        description = "N/A"
                        if os.path.exists(hp_file):
                            with open(hp_file, 'r', encoding='utf-8') as f:
                                hp_data = json.load(f)
                                description = hp_data.get('description', "N/A")
                                
                        # Append the gathered data to our results list
                        results.append({
                            "Sub-directory": entry,
                            "Best Epoch": best_epoch,
                            "Frame F1": round(frame_f1, 4),
                            "Mean IoU": round(mean_iou, 4),
                            "Segment F1": round(segment_f1, 4),
                            "Description": description
                        })
                    else:
                        print(f"Skipping '{entry}': Missing one or more required metric columns.")
                except Exception as e:
                    print(f"Error processing '{entry}': {e}")
else:
    print(f"Directory '{experiments_dir}' not found. Please make sure it is in the same directory as this notebook.")

# Create a DataFrame for a clean table rendering in Jupyter Notebook
results_df = pd.DataFrame(results)

# Display the resulting dataframe
display(results_df)

,Sub-directory,Best Epoch,Frame F1,Mean IoU,Segment F1,Description
0,bi_mamba-01,23,0.5704,0.4738,0.3833,N/A
1,bi_mamba-02,29,0.6518,0.5011,0.4583,N/A
2,decoupled_stgcn_bimamba-01,28,0.6819,0.5289,0.5051,N/A
3,decoupled_stgcn_bimamba-02,18,0.6816,0.5363,0.4959,N/A
4,decoupled_stgcn_bimamba-03,25,0.6855,0.4206,0.4895,N/A
5,decoupled_stgcn_bimamba-04,28,0.6408,0.6243,0.4142,N/A
6,decoupled_stgcn_bimamba-05,23,0.6456,0.6280,0.4303,N/A
7,decoupled_stgcn_mamba-01,25,0.6797,0.5342,0.4961,N/A
8,decoupled_stgcn_mamba-02,28,0.6761,0.5233,0.4884,N/A
9,decoupled_stgcn_mamba-03,18,0.6760,0.3984,0.4660,N/A


In [4]:
# Define the path to the experiments directory
experiments_dir = "experiments"

# --- NEW: FILTER OPTION ---
# Put the exact model_name you want to filter by (e.g., "STGCN_Mamba", "BiMamba").
# If you want to process ALL directories, leave this as None.
# The script expects the folder format to be: {model_name}-{postfix}
target_model_name = "stgcn_mamba"  # Example: "stgcn_mamba" or "bimamba"
# --------------------------

# List to store the results for each sub-directory
results = []

# Ensure the experiments directory exists before running
if os.path.exists(experiments_dir):
    # Iterate through each item in the experiments directory
    for entry in os.listdir(experiments_dir):
        
        # --- NEW: Check if the directory matches our selected model ---
        if target_model_name is not None:
            # We append "-" to ensure we match exactly the model part before the postfix
            if not entry.startswith(f"{target_model_name}-"):
                continue # Skip this folder and move to the next one
                
        sub_dir = os.path.join(experiments_dir, entry)
        
        # Check if the current item is actually a directory
        if os.path.isdir(sub_dir):
            metrics_file = os.path.join(sub_dir, "training_metrics.csv")
            hp_file = os.path.join(sub_dir, "hyperparameters.json")
            
            # Check if the metrics CSV exists in this sub-directory
            if os.path.exists(metrics_file):
                try:
                    df = pd.read_csv(metrics_file)
                    
                    # Verify that the required columns exist
                    required_cols = ['frame_f1', 'mean_iou', 'segment_f1']
                    if all(col in df.columns for col in required_cols):
                        
                        # Determine the best epoch by summing the 3 metrics
                        df['combined_score'] = df['frame_f1'] + df['mean_iou'] + df['segment_f1']
                        
                        # Find the index of the row with the highest combined score
                        best_idx = df['combined_score'].idxmax()
                        best_row = df.iloc[best_idx]
                        
                        # Get the epoch number (Assumes an 'epoch' column exists, otherwise uses row index + 1)
                        best_epoch = int(best_row['epoch']) if 'epoch' in df.columns else best_idx + 1
                        
                        frame_f1 = best_row['frame_f1']
                        mean_iou = best_row['mean_iou']
                        segment_f1 = best_row['segment_f1']
                        
                        # Look for the description in hyperparameters.json
                        description = "N/A"
                        if os.path.exists(hp_file):
                            with open(hp_file, 'r', encoding='utf-8') as f:
                                hp_data = json.load(f)
                                description = hp_data.get('description', "N/A")
                                
                        # Append the gathered data to our results list
                        results.append({
                            "Sub-directory": entry,
                            "Best Epoch": best_epoch,
                            "Frame F1": round(frame_f1, 4),
                            "Mean IoU": round(mean_iou, 4),
                            "Segment F1": round(segment_f1, 4),
                            "Description": description
                        })
                    else:
                        print(f"Skipping '{entry}': Missing one or more required metric columns.")
                except Exception as e:
                    print(f"Error processing '{entry}': {e}")
else:
    print(f"Directory '{experiments_dir}' not found. Please make sure it is in the same directory as this notebook.")

# Create a DataFrame for a clean table rendering in Jupyter Notebook
results_df = pd.DataFrame(results)

# Display the resulting dataframe (if the list isn't empty)
if not results_df.empty:
    display(results_df)
else:
    print("No results found. Either the directory is empty, or no folders matched your 'target_model_name'.")

,Sub-directory,Best Epoch,Frame F1,Mean IoU,Segment F1,Description
0,stgcn_mamba-01,26,0.6007,0.5653,0.4800,N/A
1,stgcn_mamba-02,14,0.6695,0.5373,0.4875,N/A
2,stgcn_mamba-03,19,0.6748,0.5311,0.4921,N/A
3,stgcn_mamba-04,21,0.6780,0.4021,0.4700,N/A
4,stgcn_mamba-05,29,0.6382,0.6249,0.4107,N/A
5,stgcn_mamba-06,21,0.6458,0.6131,0.4332,N/A
6,stgcn_mamba-07,29,0.6544,0.6699,0.4958,N/A
7,stgcn_mamba-08,28,0.5462,0.2767,0.0233,N/A
8,stgcn_mamba-09,27,0.5539,0.4477,0.1743,N/A
9,stgcn_mamba-10,20,0.5527,0.4848,0.1872,N/A


In [5]:
# Define the path to the experiments directory
experiments_dir = "experiments"

# --- FILTER OPTIONS ---
# Put the exact model_name you want to filter by (e.g., "stgcn_mamba", "bimamba").
# If you want to process ALL directories, leave this as None.
target_model_name = "stgcn_mamba"  # Example: "stgcn_mamba" or "bimamba"

# --- NEW: MULTIPLE HYPERPARAMETERS DISPLAY OPTION ---
# Use a list to define all the hyperparameters you want to display as separate columns.
# If you don't want to display any, just leave it as an empty list: []
target_hyperparameters = ["window_size", "overlap", "tolerance_window"] 
# ----------------------------------------------------

# List to store the results for each sub-directory
results = []

# Ensure the experiments directory exists before running
if os.path.exists(experiments_dir):
    # Iterate through each item in the experiments directory
    for entry in os.listdir(experiments_dir):
        
        # Check if the directory matches our selected model
        if target_model_name is not None:
            if not entry.startswith(f"{target_model_name}-"):
                continue # Skip this folder and move to the next one
                
        sub_dir = os.path.join(experiments_dir, entry)
        
        # Check if the current item is actually a directory
        if os.path.isdir(sub_dir):
            metrics_file = os.path.join(sub_dir, "training_metrics.csv")
            hp_file = os.path.join(sub_dir, "hyperparameters.json")
            
            # Extract multiple hyperparameters early
            description = "N/A"
            # Initialize a dictionary with "N/A" for all requested hyperparameters
            hp_values = {hp: "N/A" for hp in target_hyperparameters}
            
            if os.path.exists(hp_file):
                try:
                    with open(hp_file, 'r', encoding='utf-8') as f:
                        hp_data = json.load(f)
                        description = hp_data.get('description', "N/A")
                        
                        # Loop through the list and grab each targeted hyperparameter
                        for hp in target_hyperparameters:
                            hp_values[hp] = hp_data.get(hp, "N/A")
                except json.JSONDecodeError:
                    print(f"Warning: Could not parse {hp_file}")
            
            # Check if the metrics CSV exists in this sub-directory
            if os.path.exists(metrics_file):
                try:
                    df = pd.read_csv(metrics_file)
                    
                    # Verify that the required columns exist
                    required_cols = ['frame_f1', 'mean_iou', 'segment_f1']
                    if all(col in df.columns for col in required_cols):
                        
                        # Determine the best epoch by summing the 3 metrics
                        df['combined_score'] = df['frame_f1'] + df['mean_iou'] + df['segment_f1']
                        df['average_score'] = df['combined_score'] / 3
                        
                        # Find the index of the row with the highest combined score
                        best_idx = df['combined_score'].idxmax()
                        best_row = df.iloc[best_idx]
                        
                        # Get the epoch number (Assumes an 'epoch' column exists, otherwise uses row index + 1)
                        best_epoch = int(best_row['epoch']) if 'epoch' in df.columns else best_idx + 1
                        
                        frame_f1 = best_row['frame_f1']
                        mean_iou = best_row['mean_iou']
                        segment_f1 = best_row['segment_f1']
                        average_score = best_row['average_score']
                            
                        # Build the base dictionary for this row
                        # --- FIX: Format as strict strings to stop Pandas from auto-padding zeros ---
                        result_dict = {
                            "Sub-directory": entry,
                            "Best Epoch": best_epoch,
                            "Frame F1": f"{frame_f1:.3f}",
                            "Mean IoU": f"{mean_iou:.3f}",
                            "Segment F1": f"{segment_f1:.3f}",
                            "Average score": f"{average_score:.3f}"
                        }
                        
                        # Dynamically add EACH target hyperparameter to the table
                        for hp in target_hyperparameters:
                            result_dict[f"HP: {hp}"] = hp_values[hp]
                            
                        # Add description at the very end so it sits on the far right of the table
                        result_dict["Description"] = description                            
                            
                        results.append(result_dict)
                    else:
                        print(f"Skipping '{entry}': Missing one or more required metric columns.")
                except Exception as e:
                    print(f"Error processing '{entry}': {e}")
else:
    print(f"Directory '{experiments_dir}' not found. Please make sure it is in the same directory as this notebook.")

# Create a DataFrame for a clean table rendering in Jupyter Notebook
results_df = pd.DataFrame(results)

# Display the resulting dataframe WITHOUT the index AND with a colored column
if not results_df.empty:
    
    # Define the color you want for the Average score column here
    highlight_color = 'brown' 
    
    try:
        # For newer versions of Pandas (>= 1.4.0)
        styled_df = results_df.style.hide(axis="index").set_properties(
            **{'background-color': highlight_color}, subset=['Average score']
        )
        display(styled_df)
    except AttributeError:
        # Fallback for older versions of Pandas
        styled_df = results_df.style.hide_index().set_properties(
            **{'background-color': highlight_color}, subset=['Average score']
        )
        display(styled_df)
else:
    print("No results found. Either the directory is empty, or no folders matched your criteria.")

Sub-directory,Best Epoch,Frame F1,Mean IoU,Segment F1,Average score,HP: window_size,HP: overlap,HP: tolerance_window,Description
stgcn_mamba-01,26,0.601,0.565,0.480,0.549,1000,200,N/A,N/A
stgcn_mamba-02,14,0.669,0.537,0.488,0.565,1000,200,5,N/A
stgcn_mamba-03,19,0.675,0.531,0.492,0.566,1000,200,5,N/A
stgcn_mamba-04,21,0.678,0.402,0.470,0.517,1000,200,5,N/A
stgcn_mamba-05,29,0.638,0.625,0.411,0.558,1000,200,5,N/A
stgcn_mamba-06,21,0.646,0.613,0.433,0.564,1000,200,5,N/A
stgcn_mamba-07,29,0.654,0.670,0.496,0.607,512,200,5,N/A
stgcn_mamba-08,28,0.546,0.277,0.023,0.282,4096,0,1,N/A
stgcn_mamba-09,27,0.554,0.448,0.174,0.392,2048,0,1,N/A
stgcn_mamba-10,20,0.553,0.485,0.187,0.408,2048,0,1,N/A
